In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
import nltk
nltk.download('vader_lexicon')

# Load dataset
df = pd.read_csv("train_raw.csv")
df.head()

In [ ]:
def extract_experience(prompt):
    match = re.search(r"(\d+)[ ]*years of experience", prompt)
    return int(match.group(1)) if match else None

df['Extracted Experience'] = df['Prompt'].apply(extract_experience)
df['Experience Match'] = df['Extracted Experience'] == df['Years of Experience']

def extract_snomed_codes(snomed_str):
    if pd.isna(snomed_str):
        return []
    codes = re.findall(r"(\d{6,})", snomed_str)
    return list(set(codes))

df['DDX_Codes'] = df['DDX SNOMED'].apply(extract_snomed_codes)
df['Num_DDX_Codes'] = df['DDX_Codes'].apply(len)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
clinician_embeddings = model.encode(df['Clinician'].astype(str).tolist())
gpt4_embeddings = model.encode(df['GPT4.0'].astype(str).tolist())
df['GPT4_vs_Clinician_Similarity'] = [cosine_similarity([a], [b])[0][0] for a, b in zip(clinician_embeddings, gpt4_embeddings)]

In [ ]:
sia = SentimentIntensityAnalyzer()
df['Clinician_Sentiment'] = df['Clinician'].apply(lambda x: sia.polarity_scores(str(x))['compound'])
df['GPT4_Sentiment'] = df['GPT4.0'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

## 📊 Exploratory Data Analysis (EDA)

In [ ]:
# County distribution
plt.figure(figsize=(12,6))
df['County'].value_counts().plot(kind='bar')
plt.title("Cases by County")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

# Experience distribution
sns.histplot(df['Years of Experience'], kde=True)
plt.title("Nurse Experience Distribution")
plt.xlabel("Years")
plt.show()

# Health level
sns.countplot(data=df, y='Health level', order=df['Health level'].value_counts().index)
plt.title("Health Facility Types")
plt.show()

# Competency
sns.countplot(data=df, y='Nursing Competency', order=df['Nursing Competency'].value_counts().index)
plt.title("Nursing Competency Areas")
plt.show()

# Word cloud
text = " ".join(df['Prompt'].astype(str))
wordcloud = WordCloud(width=800, height=400).generate(text)
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Prompts")
plt.show()

## 📊 Similarity & Sentiment Visuals

In [ ]:
sns.histplot(df['GPT4_vs_Clinician_Similarity'], kde=True)
plt.title("Semantic Similarity: GPT-4 vs Clinician")
plt.xlabel("Cosine Similarity")
plt.show()

sns.countplot(data=df, x='Experience Match')
plt.title("Experience Mismatch (Prompt vs Metadata)")
plt.show()

sns.boxplot(data=df[['Clinician_Sentiment', 'GPT4_Sentiment']])
plt.title("Sentiment Comparison")
plt.ylabel("Compound Score")
plt.show()

## 📎 SNOMED Diagnostic Patterns

In [ ]:
plt.hist(df['Num_DDX_Codes'], bins=10)
plt.title("Distribution of Diagnosis Codes")
plt.xlabel("# SNOMED Codes")
plt.ylabel("Cases")
plt.show()

## 🔍 Manual Case Review

In [ ]:
sample = df.sample(1).iloc[0]
print("--- Prompt:\n", sample['Prompt'])
print("\n--- Clinician:\n", sample['Clinician'])
print("\n--- GPT-4:\n", sample['GPT4.0'])

## 📈 Results Summary

In [ ]:
print("""
Key Results:
- Most prompts correctly match the structured experience metadata.
- GPT-4 responses show strong semantic alignment (similarity ~0.85 avg).
- Sentiment varies—GPT-4 often less cautious or empathetic.
- SNOMED parsing reveals 2–5 expected conditions per prompt.
""")

## 💬 Discussion

In [ ]:
print("""
- AI models can reasonably approximate clinician reasoning semantically.
- They may diverge in tone and diagnostic confidence.
- Clinician trust and patient safety require factual correctness, not just fluency.
""")

## ✅ Recommendations

In [ ]:
print("""
- AI responses should be reviewed before clinical use.
- Highlight hallucinations and unsafe guidance in evaluations.
- Incorporate regional clinical guidelines and SNOMED linking in training.
""")

## 🔮 Future Work

In [ ]:
print("""
- Extend model to LLAMA and GEMINI.
- Incorporate BLEU/ROUGE for lexical comparison.
- Implement topic modeling (e.g., BERTopic) for vignette clustering.
- Develop a simple RAG-based chatbot for decision support.
""")

## 📚 Appendix

In [ ]:
print("""
Appendix:
- SNOMED glossary (external reference)
- Manual vignette examples
- Experience parser function
- Prompt vs metadata audit table
""")